<h1> Ridge Regression

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score, mean_squared_error

source_file_path = 'data/co2_source.xlsx'
source_sheets = ['Coal', 'Natural gas', 'Petroleum']

sector_file_path = 'data/co2_sector.xlsx'
sector_sheets = ['Residential', 'Commercial', 'Industrial', 'Transportation', 'Electric power', 'Total']

def load_excel_data(path, sheet_list):
    merged_df = None
    for sheet in sheet_list:
        
        df = pd.read_excel(path, sheet_name=sheet, skiprows=2)
        
        
        df_melt = df.melt(id_vars=['State'], var_name='Year', value_name=sheet)
        df_melt['Year'] = pd.to_numeric(df_melt['Year'], errors='coerce')
        
        if merged_df is None:
            merged_df = df_melt
        else:
            merged_df = pd.merge(merged_df, df_melt, on=['State', 'Year'], how='outer')
            
    
    merged_df = merged_df.dropna(subset=['Year'])
    merged_df['Year'] = merged_df['Year'].astype(int)
    merged_df = merged_df[merged_df['State'] != 'US'].dropna()
    return merged_df

df_sector = load_excel_data(sector_file_path, sector_sheets)
df_source = load_excel_data(source_file_path, source_sheets)

df_clean = pd.merge(df_sector, df_source, on=['State', 'Year'], how='inner')

X = df_clean[['Year', 'State']]
y = df_clean['Total']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['Year']),
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['State'])
    ])

ridge_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RidgeCV(alphas=np.logspace(-3, 3, 20)))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
ridge_pipeline.fit(X_train, y_train)

y_pred = ridge_pipeline.predict(X_test)
print(f"Model R-squared: {r2_score(y_test, y_pred):.4f}")
print(f"Best Alpha: {ridge_pipeline.named_steps['regressor'].alpha_:.4f}")

#Test Prediction
test_case = pd.DataFrame({'Year': [2025], 'State': ['TX']})
pred_val = ridge_pipeline.predict(test_case)
print(f"\nPredicted 2025 Total CO2 for TX: {pred_val[0]:.2f} Million Metric Tons")

<h3> Multiple Output Ridge Regression
<h4> Predicts Total Using Sum of All Sectors

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.multioutput import MultiOutputRegressor

source_file_path = 'data/co2_source.xlsx'
source_sheets = ['Coal', 'Natural gas', 'Petroleum']

sector_file_path = 'data/co2_sector.xlsx'
sector_sheets = ['Residential', 'Commercial', 'Industrial', 'Transportation', 'Electric power', 'Total']

def load_excel_data(path, sheet_list):
    merged_df = None
    for sheet in sheet_list:
        
        df = pd.read_excel(path, sheet_name=sheet, skiprows=2)
        
        
        df_melt = df.melt(id_vars=['State'], var_name='Year', value_name=sheet)
        df_melt['Year'] = pd.to_numeric(df_melt['Year'], errors='coerce')
        
        if merged_df is None:
            merged_df = df_melt
        else:
            merged_df = pd.merge(merged_df, df_melt, on=['State', 'Year'], how='outer')
            
    
    merged_df = merged_df.dropna(subset=['Year'])
    merged_df['Year'] = merged_df['Year'].astype(int)
    merged_df = merged_df[merged_df['State'] != 'US'].dropna()
    return merged_df

df_sector = load_excel_data(sector_file_path, sector_sheets)
df_source = load_excel_data(source_file_path, source_sheets)

df_clean = pd.merge(df_sector, df_source, on=['State', 'Year'], how='inner')

X = df_clean[['Year', 'State']]
y = df_clean[['Coal', 'Natural gas', 'Petroleum']]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['Year']),
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['State'])
    ])

ridge_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', MultiOutputRegressor(RidgeCV(alphas=np.logspace(-3, 3, 20))))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
ridge_pipeline.fit(X_train, y_train)

y_pred = ridge_pipeline.predict(X_test)

for i, target in enumerate(y.columns):
    print(f"Model R-squared for {target}: {r2_score(y_test[target], y_pred[:, i]):.4f}")

print(f"Best Alpha: {ridge_pipeline.named_steps['regressor'].estimators_[0].alpha_:.4f}")

#Test Prediction
test_case = pd.DataFrame({'Year': [2025], 'State': ['TX']})
pred_val = ridge_pipeline.predict(test_case)

print("\nPredicted 2025 Outputs for TX:")
print(f"Total CO2: {pred_val.sum(axis=1)[0]:.2f} Million Metric Tons ")
for i, target in enumerate(y.columns):
    sector_total_sum += pred_val[0, i]
    print(f"{target}: {pred_val[0, i]:.2f} Million Metric Tons")

<h3> Ridge Regression per State
<h4> Predicts total by only training on a single state's data

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score, mean_squared_error

source_file_path = 'data/co2_source.xlsx'
source_sheets = ['Coal', 'Natural gas', 'Petroleum']

sector_file_path = 'data/co2_sector.xlsx'
sector_sheets = ['Residential', 'Commercial', 'Industrial', 'Transportation', 'Electric power', 'Total']

def load_excel_data(path, sheet_list):
    merged_df = None
    for sheet in sheet_list:
        
        df = pd.read_excel(path, sheet_name=sheet, skiprows=2)
        
        
        df_melt = df.melt(id_vars=['State'], var_name='Year', value_name=sheet)
        df_melt['Year'] = pd.to_numeric(df_melt['Year'], errors='coerce')
        
        if merged_df is None:
            merged_df = df_melt
        else:
            merged_df = pd.merge(merged_df, df_melt, on=['State', 'Year'], how='outer')
            
    
    merged_df = merged_df.dropna(subset=['Year'])
    merged_df['Year'] = merged_df['Year'].astype(int)
    merged_df = merged_df[merged_df['State'] != 'US'].dropna()
    return merged_df

df_sector = load_excel_data(sector_file_path, sector_sheets)
df_source = load_excel_data(source_file_path, source_sheets)

df_clean = pd.merge(df_sector, df_source, on=['State', 'Year'], how='inner')

states = df_clean['State'].unique()

predictions = {}

for state in states:
    # Filter data for the current state
    df_state = df_clean[df_clean['State'] == state].copy()
    
    # Features and target
    X = df_state[['Year']]
    y = df_state['Total']
    
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), ['Year'])
        ])


    ridge_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', RidgeCV(alphas=np.logspace(-3, 3, 20)))
    ])
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    ridge_pipeline.fit(X_train, y_train)
    
    test_case = pd.DataFrame({'Year': [2025]})
    pred_val = ridge_pipeline.predict(test_case)
    predictions[state] = pred_val[0]

# Print predictions
for state, pred in predictions.items():
    print(f"Predicted 2025 Total CO2 for {state}: {pred:.2f} Million Metric Tons")